In [1]:
# 1. Import monkey-patched read_text and check CUDA availability
import pathlib
orig = pathlib.Path.read_text
pathlib.Path.read_text = lambda self, encoding=None, errors=None: orig(self, encoding='utf-8' if encoding is None else encoding, errors=errors)

import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CUDA is NOT available. Running SFT training will be extremely slow on CPU.")

PyTorch version: 2.5.1+cu121
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU


c:\Users\mduy\source\repos\EXACT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 2. Inspect Preprocessed Data Samples
from llm.tuning.train import load_logic_dataset
import os

# Resolve paths
logic_path = "../../../data/processed/logic_based.json"
if not os.path.exists(logic_path):
    logic_path = "../../../data/logic_based.json"

# Check logic data
if os.path.exists(logic_path):
    logic_samples = load_logic_dataset(logic_path)
    print("\n--- Sample Logic Prompt/Response ---")
    sample_msg = logic_samples[0]["messages"]
    for msg in sample_msg:
        print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

Loading logic dataset from: ../../../data/processed/logic_based.json
Loaded 808 logical reasoning training instances.

--- Sample Logic Prompt/Response ---
[SYSTEM]:
You are a logical reasoning assistant. Solve the user's question by outputting a JSON object with 'answer' and 'explanation' fields.

[USER]:
Context (Premises):
- If a Python code is well-tested, then the project is optimized.
- If a Python code does not follow PEP 8 standards, then it is not well-tested.
- All Python projects are easy to maintain.
- All Python code is well-tested.
- If a Python code follows PEP 8 standards, then it is easy to maintain.
- If a Python code is well-tested, then it follows PEP 8 standards.
- If a Python project is well-structured, then it is optimized.
- If a Python project is easy to maintain, then it is well-tested.
- If a Python project is optimized, then it has clean and readable code.
- All Python projects are well-structured.
- All Python projects have clean and readable code.
- There 

In [3]:
from regex import B
from llm.tuning.train import load_logic_dataset
# 3. Run SFT Training
# You can customize the parameters by modifying the sys.argv simulation below.
from llm.tuning.train import main as run_pipeline
import sys

# Set custom training parameters
sys.argv = [
    "train.py",
    "--model_id", "Qwen/Qwen2.5-7B-Instruct", # Model to use
    "--dataset_type", "both",                   # Dataset to train on: 'logic', 'physics', or 'both'
    "--epochs", "3",                            # Number of training epochs
    "--batch_size", "1",                        # Keep at 1 for 8GB VRAM RTX 4060
    "--grad_accum", "8",                        # Effective batch size = batch_size * grad_accum = 8
    "--lr", "2e-4",
    "--max_length", "512",                      # Maximum context size
    "--output_dir", "./results"                 # Output directory for saving model adapters
]

run_pipeline()

Loading logic dataset from: C:\Users\mduy\source\repos\EXACT\data\processed\logic_based.json
Loaded 808 logical reasoning training instances.
Loading physics dataset from: C:\Users\mduy\source\repos\EXACT\data\physic.csv
Loaded 1352 physics training instances.
Total training dataset size: 2160 instances.
Initializing model and tokenizer for Qwen/Qwen2.5-7B-Instruct...


Map: 100%|██████████| 2160/2160 [00:00<00:00, 25558.57 examples/s]


Train split size: 1944, Val split size: 216
Loading base model to device: auto


Loading weights: 100%|██████████| 339/339 [00:12<00:00, 26.69it/s]


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


Tokenizing eval dataset: 100%|██████████| 216/216 [00:00<00:00, 1693.93 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting SFT training pipeline...


KeyboardInterrupt: 

In [ ]:
# 4. Test Fine-tuned Model Inference
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json

model_id = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "./results"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model {model_id} and adapter weights from {adapter_path}...")
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None
)
model = PeftModel.from_pretrained(base_model, adapter_path).to(device)
model.eval()

# Define testing prompt
test_prompt = "Calculate the equivalent resistance of three parallel resistors, each of resistance r."
system_prompt = "You are a physics assistant. Solve the user's question by outputting a JSON object with 'answer' and 'explanation' fields."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(inputs, max_new_tokens=256, eos_token_id=tokenizer.eos_token_id)

response_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
print("\n--- Model Generated Output (Expected JSON format) ---")
print(response_text)
print("----------------------------------------------------")